# VD20 — can the volatility engine forecast the next 24 hours, and does that beat implied?

Protocol: `discussions/Volatility_Dynamics/040_claude_vd20_next_24h_volatility_preregistration.md`.

Origin 08:00 UTC, target = realised variance over the next 24h (Σ r², 5m). Models: NAIVE (last 24h), ENGINE (VD18 HAR_4 = level of AM-T-HAR-Blend), HAR_D (1/5/22-day HAR), HAR_DX (HAR_D + intraday lags + leverage).
**G1** best of ENGINE/HAR_D beats NAIVE (QLIKE). **G2** ENGINE vs HAR_D. **G3** on VD19's 12:00 option rows: does the forecast add to implied?

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14, markov_har_variance as v18, model_vs_implied as mv, daily_vol_forecast as dv
for m in (v14,v18,mv,dv):importlib.reload(m)
SYNTHETIC=os.getenv('VD20_SYNTHETIC','0')=='1'
P=dv.Params(n_boot=400) if SYNTHETIC else dv.Params()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='vd20-');OUT=Path(tmp.name)/'20'
else:
    OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'20'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_daily_vol_forecast.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-400:]);assert chk.returncode==0,'VD20 tests failed'
print(json.dumps(dv.registration(P),indent=1,default=str))

.....                                                                    [100%]
5 passed in 6.87s

{
 "version": "VD20-0.1.0",
 "forecast_end": "2026-10-15T00:00:00Z",
 "min_frac_daily": 0.98,
 "vol_quantile": 0.8,
 "tail_k": [
  1.0,
  1.5,
  2.0,
  2.5
 ],
 "trade_ratio": 1.25,
 "n_boot": 4000,
 "seed": 2010,
 "status": "HISTORICAL_EXPLORATORY_PREREGISTERED_VD040",
 "origins": {
  "8": 1440,
  "12": 1200
 },
 "primary_hour": 8,
 "assets": [
  "AVAX",
  "BTC",
  "ETH",
  "SOL",
  "XRP"
 ],
 "models": [
  "NAIVE",
  "ENGINE",
  "HAR_D",
  "HAR_DX"
 ],
 "daily_designs": {
  "HAR_D": [
   "l1",
   "l5",
   "l22"
  ],
  "HAR_DX": [
   "l1",
   "l5",
   "l22",
   "l30",
   "l60",
   "l240",
   "neg"
  ]
 }
}


## Engine refit (VD19 code, unchanged) and daily origins

In [2]:
T0=time.time();H=sorted(set(dv.ORIGINS.values()))
if SYNTHETIC:
    raw_all=dv.synthetic_returns(n_days=520,assets=('BTC','ETH','SOL'),seed=11,start='2025-04-01')
    raw_fit=raw_all[raw_all.state_time<pd.Timestamp('2026-08-01',tz='UTC')];ASSETS=['BTC','ETH','SOL']
else:
    HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
    raw_fit,excl=v14.load_returns(HIST,mv.FIT_ASSETS,end=pd.Timestamp(mv.Params().fit_sample_end))
    raw_all,_=v14.load_returns(HIST,dv.ASSETS,end=pd.Timestamp(P.forecast_end),min_days=365);ASSETS=list(dv.ASSETS)
    print('excluded from engine fit:',excl)
frame=mv.build_frame(raw_fit,H,v14,v18)
FITS=mv.fit_har_models(frame,H,v18)
TRAIN_END=pd.Timestamp(frame.loc[frame.split=='train','date'].max())
print('engine HAR_4 fitted at',H,'| TRAIN ends',TRAIN_END.date(),'| fit universe',sorted(frame.underlying.unique()))
del frame,raw_fit
PANEL=mv.forecast_panel(raw_all,v14,v18)
Ds={}
for hr,h in dv.ORIGINS.items():
    D=dv.daily_origins(raw_all,hr,h,P)
    o=D[['underlying','state_time']].assign(h=h)
    D['F_ENGINE']=mv.model_sigma(PANEL,o,FITS,v18).to_numpy()**2
    Ds[hr]=D
D8=Ds[dv.PRIMARY_HOUR]
tr8=D8[D8.date+pd.Timedelta(days=1)<=TRAIN_END]
FD={hr:dv.fit_daily(Ds[hr][Ds[hr].date+pd.Timedelta(days=1)<=TRAIN_END]) for hr in Ds}
for hr in Ds:Ds[hr]=pd.concat([Ds[hr],dv.predict_daily(Ds[hr],FD[hr])],axis=1)
D8=Ds[dv.PRIMARY_HOUR]
print({m:{k:np.round(v,3) for k,v in zip(['const',*FD[8][m]['cols']],FD[8][m]['beta'])} for m in dv.DAILY})
print(f'{time.time()-T0:.0f}s')

excluded from engine fit: {'HYPE': 'insufficient history'}
engine HAR_4 fitted at [1200, 1440] | TRAIN ends 2025-11-22 | fit universe ['AVAX', 'BTC', 'ETH', 'SOL', 'TRX', 'XRP']
{'HAR_D': {'const': np.float64(-1.997), 'l1': np.float64(0.469), 'l5': np.float64(0.17), 'l22': np.float64(0.044)}, 'HAR_DX': {'const': np.float64(-0.47), 'l1': np.float64(0.098), 'l5': np.float64(0.179), 'l22': np.float64(0.036), 'l30': np.float64(0.036), 'l60': np.float64(0.033), 'l240': np.float64(0.374), 'neg': np.float64(-0.06)}}
21s


## G1/G2 — forecast accuracy over the next 24h (EVAL = after the engine's TRAIN)

In [3]:
EV=D8[D8.date>TRAIN_END].copy();TR=D8[D8.date+pd.Timedelta(days=1)<=TRAIN_END].copy()
SC,E=dv.score_table(EV,list(dv.MODELS),P);display(SC.round(4))
PAIRS=[('ENGINE','NAIVE'),('HAR_D','NAIVE'),('HAR_DX','NAIVE'),('ENGINE','HAR_D'),('HAR_DX','HAR_D')]
PD=dv.pair_diffs(E,PAIRS,P);display(PD.round(4))
print('Per asset QLIKE:');display(E.assign(**{m:dv.qlike(E.rv_next,E[f'F_{m}']) for m in dv.MODELS}).groupby('underlying')[list(dv.MODELS)].mean().round(4))
E['period']=np.where(E.date<pd.Timestamp('2026-09-19',tz='UTC'),'pre_2026-09-19','post (live collector era)')
print('By period:');display(E.assign(**{m:dv.qlike(E.rv_next,E[f'F_{m}']) for m in dv.MODELS}).groupby('period')[list(dv.MODELS)].agg(['mean','size']).round(4))

,model,n,days,qlike,qlike_lo,qlike_hi,mse_log,mz_a,mz_b,mz_r2,bias_mean_ratio
0,NAIVE,1625,325,0.3553,0.2822,0.4433,0.5903,-2.6086,0.6415,0.4106,0.9961
1,ENGINE,1625,325,0.2002,0.1723,0.2318,0.4276,-0.8487,0.9054,0.5243,0.9750
2,HAR_D,1625,325,0.2960,0.2634,0.3340,0.8443,-2.2376,0.7510,0.4145,0.6321
3,HAR_DX,1625,325,0.2603,0.2343,0.2893,0.6929,-2.1777,0.7504,0.4988,0.6352


,model,vs,qlike_diff,lo,hi,n,better
0,ENGINE,NAIVE,-0.1550,-0.2231,-0.0946,1625,model
1,HAR_D,NAIVE,-0.0592,-0.1392,0.0099,1625,tie
2,HAR_DX,NAIVE,-0.0950,-0.1718,-0.0291,1625,model
3,ENGINE,HAR_D,-0.0958,-0.1195,-0.0713,1625,model
4,HAR_DX,HAR_D,-0.0357,-0.0541,-0.0192,1625,model


Per asset QLIKE:


,NAIVE,ENGINE,HAR_D,HAR_DX
underlying,,,,
AVAX,0.2510,0.1708,0.4770,0.4772
BTC,0.4252,0.2214,0.2232,0.1846
ETH,0.4304,0.2166,0.2429,0.2046
SOL,0.3234,0.1794,0.2199,0.1749
XRP,0.3462,0.2130,0.3171,0.2601


By period:


NAIVE        ENGINE         HAR_D        HAR_DX      
                             mean  size    mean  size    mean  size    mean  size
period                                                                           
post (live collector era)  0.3655   125  0.1640   125  0.2792   125  0.2341   125
pre_2026-09-19             0.3544  1500  0.2033  1500  0.2974  1500  0.2625  1500

## Volatile-day classification and tails (what an OTM buyer needs)

In [4]:
CL=dv.classification(E,list(dv.MODELS),P);display(CL.round(3))
TRs=pd.concat([TR],axis=1);TT=dv.tail_table(TRs,E,list(dv.MODELS),P)
display(TT.pivot_table(index='k',columns='model',values=['observed','gaussian','student_t_train']).round(4))
best=SC.sort_values('qlike').iloc[0].model
E['dec']=pd.qcut(np.log(E[f'F_{best}']/E.groupby('underlying').rv_next.transform('median')),10,labels=False)
print(f'Next-24h |move| by decile of {best} forecast (relative to asset median RV):')
display(E.groupby('dec').apply(lambda g:pd.Series({'n':len(g),'pred_sd_pct':100*np.sqrt(g[f'F_{best}']).mean(),
    'real_sd_pct':100*np.sqrt(g.rv_next).mean(),'abs_move_pct':100*g.R_next.abs().mean(),
    'P|z|>1.5':(g.R_next.abs()>1.5*np.sqrt(g[f'F_{best}'])).mean()})).round(3))

,model,n,base_rate,auc,auc_lo,auc_hi,hit_top_decile,lift_top_decile,hit_bottom_decile
0,NAIVE,1625,0.2,0.828,0.782,0.872,0.712,3.558,0.037
1,ENGINE,1625,0.2,0.840,0.794,0.882,0.706,3.528,0.012
2,HAR_D,1625,0.2,0.774,0.732,0.813,0.497,2.485,0.025
3,HAR_DX,1625,0.2,0.790,0.747,0.828,0.564,2.822,0.025


gaussian                         observed                         student_t_train                        
model   ENGINE   HAR_D  HAR_DX   NAIVE   ENGINE   HAR_D  HAR_DX   NAIVE          ENGINE   HAR_D  HAR_DX   NAIVE
k                                                                                                              
1.0     0.3173  0.3173  0.3173  0.3173   0.2455  0.1815  0.1957  0.2843          0.2712  0.2046  0.2071  0.2831
1.5     0.1336  0.1336  0.1336  0.1336   0.1034  0.0714  0.0708  0.1465          0.1331  0.0903  0.0927  0.1470
2.0     0.0455  0.0455  0.0455  0.0455   0.0578  0.0283  0.0326  0.0757          0.0694  0.0441  0.0459  0.0820
2.5     0.0124  0.0124  0.0124  0.0124   0.0246  0.0111  0.0148  0.0412          0.0389  0.0237  0.0251  0.0493

Next-24h |move| by decile of ENGINE forecast (relative to asset median RV):


,n,pred_sd_pct,real_sd_pct,abs_move_pct,P|z|>1.5
dec,,,,,
0,163.0,1.633,1.787,1.329,0.117
1,162.0,2.202,2.171,1.817,0.148
2,163.0,2.401,2.278,1.846,0.141
3,162.0,2.668,2.508,1.843,0.130
4,163.0,2.865,2.559,1.975,0.092
5,162.0,3.062,2.754,1.877,0.043
6,162.0,3.294,3.007,2.113,0.074
7,163.0,3.585,3.448,2.669,0.110
8,162.0,3.898,3.835,2.552,0.074


## G3 — versus implied: VD19 12:00 entries (1200 min to the 08:00 expiry)

In [5]:
MODELS3=list(dv.MODELS)
if SYNTHETIC:
    d,sm,rv=mv.synthetic_comparison(n_sessions=25,seed=5,model_skill=0.6)
    R19=pd.DataFrame({'session':d.session,'underlying':d.underlying,'entry_hour':12,'sigma_imp':d.iv_straddle_mid*np.sqrt(d['T']),'rv_path':rv,'R_net':d.get('R_net',0.0)})
    D12=Ds[12].copy();ss=sorted(R19.session.unique());dd=sorted(D12.date.unique())[-len(ss):]
    mp=dict(zip(dd,[pd.Timestamp(s,tz='UTC')-pd.Timedelta(days=1) for s in ss]));D12['date']=D12.date.map(mp)
    D12=D12.dropna(subset=['date']);R19['underlying']=R19.underlying.map(dict(zip(sorted(R19.underlying.unique()),ASSETS)))
else:
    R19=pd.read_csv(ROOT/'notebooks'/'Volatility_Dynamics'/'19'/'19_rows.csv',dtype={'session':str});D12=Ds[12]
M3,INFO,ECON,IMP=dv.implied_check(R19,D12,MODELS3,P,mv)
print('rows',len(M3),'sessions',M3.session.nunique());print('implied alone:',{k:round(v,3) for k,v in IMP.items() if isinstance(v,float)})
display(INFO.round(3).T);display(ECON.round(3))
print('Mean log(F_model / implied variance) at 12:00:');display(M3[[f'x_{m}' for m in MODELS3]].describe().round(3))

rows 86 sessions 22
implied alone: {'const': -0.334, 'const_lo': -1.646, 'const_hi': 0.748, 'log_var_imp': 0.948, 'log_var_imp_lo': 0.768, 'log_var_imp_hi': 1.092}


,0,1,2,3
model,NAIVE,ENGINE,HAR_D,HAR_DX
enc_n,86,86,86,86
enc_sessions,22,22,22,22
enc_const,-0.5,-0.18,-0.804,-0.08
enc_const_lo,-1.71,-1.443,-1.714,-1.423
enc_const_hi,0.667,1.159,0.224,1.127
enc_log_var_imp,1.028,0.851,1.1,0.699
enc_log_var_imp_lo,0.677,0.442,0.545,0.261
enc_log_var_imp_hi,1.232,1.161,1.357,1.009
enc_log_F_NAIVE,-0.103,NaN,NaN,NaN


,model,subset,n_sessions,n,mean,lo,hi
0,NAIVE,sigma_model>=1.25 sigma_imp,11,20,-0.062,-0.334,0.201
1,ENGINE,sigma_model>=1.25 sigma_imp,15,32,-0.230,-0.467,0.010
2,HAR_D,sigma_model>=1.25 sigma_imp,18,53,-0.313,-0.488,-0.121
3,HAR_DX,sigma_model>=1.25 sigma_imp,20,59,-0.285,-0.434,-0.123
4,-,all 12:00 straddles,22,86,-0.229,-0.393,-0.050


Mean log(F_model / implied variance) at 12:00:


,x_NAIVE,x_ENGINE,x_HAR_D,x_HAR_DX
count,86.000,86.000,86.000,86.000
mean,0.061,0.145,0.309,0.296
std,0.328,0.208,0.248,0.186
min,-0.795,-0.450,-0.068,-0.184
25%,-0.069,-0.007,0.153,0.195
50%,0.048,0.131,0.268,0.301
75%,0.216,0.295,0.404,0.426
max,0.990,0.637,1.163,0.849


## Decision and save

In [6]:
DEC=dv.decide(PD,INFO);DEC['train_end']=str(TRAIN_END.date());DEC['eval_days']=int(E.date.nunique());DEC['best_qlike_model']=best
print(json.dumps(DEC,indent=1,default=str))
if not SYNTHETIC:
    SC.to_csv(OUT/'20_scores.csv',index=False);PD.to_csv(OUT/'20_pairs.csv',index=False);CL.to_csv(OUT/'20_classification.csv',index=False)
    TT.to_csv(OUT/'20_tails.csv',index=False);INFO.to_csv(OUT/'20_vs_implied.csv',index=False);ECON.to_csv(OUT/'20_economics.csv',index=False)
    E[['date','underlying','rv_next','R_next',*[f'F_{m}' for m in dv.MODELS]]].to_csv(OUT/'20_rows.csv',index=False,float_format='%.6g')
    (OUT/'20_decision.json').write_text(json.dumps(DEC,indent=1,default=str))
    (OUT/'20_manifest.json').write_text(json.dumps({'registration':dv.registration(P),'daily_fits':FD[8],'runtime_s':time.time()-T0},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')

{
 "version": "VD20-0.1.0",
 "G1_skill_vs_naive": "PASS",
 "G2_engine_vs_daily_har": "ENGINE_BETTER",
 "G3_beyond_implied": "FAIL",
 "G3_model": null,
 "verdict": "FORECASTS_VOL_BUT_NOT_BEYOND_IMPLIED",
 "train_end": "2025-11-22",
 "eval_days": 325,
 "best_qlike_model": "ENGINE"
}
total 35s
